# 01 · Datos: el problema y el conjunto de datos

Esta etapa plantea el problema, presenta el conjunto de datos (qué tablas y columnas se esperan y en
qué momento del proceso se conoce cada una) y entrega las tablas crudas con las que trabaja el resto
del proyecto, sin limpiar nada todavía.

[Notebook completo](Proyecto_Final.ipynb) &nbsp;·&nbsp; Siguiente: [02 · Exploración](02_exploracion.ipynb) →

## Código del proyecto

La celda siguiente recrea el paquete `proyecto/` dentro de la sesión, de forma que el notebook funciona en Google Colab sin clonar ningún repositorio: basta con ejecutarla una vez (*Entorno de ejecución → Ejecutar todas*). En una copia local del repositorio no reemplaza ningún archivo. La sección puede plegarse desde su título.

In [ ]:
# Código del proyecto. Esta celda recrea en el directorio de trabajo:
#   - el paquete `proyecto/` (49 módulos) y `configuracion/mapeo_columnas.json`;
# para que el notebook funcione en Google Colab sin clonar el repositorio. Todo viaja
# al final de la celda como un .tar.xz en base64 (60 archivos, 177 kB).
# Antes de extraer se comprueba su huella SHA-256. Un archivo que ya existe con el
# mismo contenido no se toca, y uno que existe con otro contenido (una copia local
# del repositorio con cambios, por ejemplo) se conserva y se avisa; para
# reemplazarlo hay que poner SOBRESCRIBIR = True.
import base64
import hashlib
import importlib
import io
import os
import sys
import tarfile
from pathlib import Path

SOBRESCRIBIR = False


def recrear_proyecto(huella, codificado):
    datos = base64.b64decode(codificado)
    if hashlib.sha256(datos).hexdigest() != huella:
        raise ValueError("El código embebido está dañado: su SHA-256 no coincide.")
    escritos, iguales, conservados = 0, 0, []
    with tarfile.open(fileobj=io.BytesIO(datos), mode="r:xz") as archivo:
        for miembro in archivo.getmembers():
            nombre = miembro.name
            if Path(nombre).is_absolute() or ".." in Path(nombre).parts:
                raise ValueError(f"Ruta no permitida en el archivo: {nombre}")
            contenido, destino = archivo.extractfile(miembro).read(), Path(nombre)
            if destino.exists() and destino.read_bytes() == contenido:
                iguales += 1
            elif destino.exists() and not SOBRESCRIBIR:
                conservados.append(nombre)
            else:
                destino.parent.mkdir(parents=True, exist_ok=True)
                destino.write_bytes(contenido)
                escritos += 1
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
    importlib.invalidate_caches()
    print(f"Código del proyecto: {escritos} archivos escritos, {iguales} ya estaban iguales.")
    if conservados:
        print(f"Se conservaron {len(conservados)} archivos locales distintos:", ", ".join(conservados))


recrear_proyecto(
    huella="a2d58283cc5d54a9eaafe99d0447eab556292aad1e7bcd59c2b58d8adb8961f3",
    codificado="...",
)

## Desarrollo de la etapa

In [1]:
from proyecto.base.artefactos import iniciar_etapa
from proyecto.base.salida import mostrar
from proyecto.etapas import datos

iniciar_etapa("01_datos")
c = datos.iniciar()
mostrar(c.encabezado, c.autores)

## 1. El problema

Un laboratorio de microbiología recibe a diario muestras de producto terminado, materia prima, agua
y muestreos en planta de clientes de los sectores cosmético, farmacéutico, de alimentos, veterinario
y de centrales de mezclas, y de cada muestra guarda quién la envió, en qué condiciones llegó, qué
ensayo se le hizo, quién la procesó y cuál fue el resultado (contexto de empresa: Rodam Análisis
SAS, Bogotá; el objeto de estudio es la base de muestras, no la empresa). Con esa base se plantean
tres preguntas:

- **Anticipar la demanda:** cuántas muestras van a llegar, para programar analistas, medios e
  incubadoras.
- **Estimar el riesgo de cada resultado:** qué tan probable es que una muestra resulte no conforme
  y, si sale positiva, que se trate de un falso positivo.
- **Entender los grupos:** qué clientes o muestras se parecen entre sí.

De cada pregunta salen varios análisis posibles, de forma que el proyecto declara un catálogo de
candidatos, los somete a la misma prueba rápida (etapa 03) y solo estudia a fondo a los ganadores.

## 2. El contrato de datos

El contrato (`proyecto/datos/esquema.py`) es la única fuente de verdad sobre los nombres de
columna: de cada una declara su tipo, su papel, si es obligatoria y el **momento** del proceso en que
el dato pasa a conocerse. Si la exportación real no trae una columna opcional, los análisis que la
usan simplemente se omiten.

El momento es lo que más adelante impide la fuga de información: cada análisis declara en qué
momento se hace la predicción, y sus variables solo pueden ser columnas conocidas hasta ese momento.

In [2]:
datos.contrato(c).mostrar()

Columna,Tipo,Papel,Momento,Obligatoria,Descripción
id_cliente,id,id,maestro,sí,Identificador del cliente
sector,categoria,variable,maestro,sí,Línea de negocio del cliente
zona,categoria,variable,maestro,no,Municipio o zona de la planta
tamano,categoria,variable,maestro,no,Tamaño de la empresa
tipo_contrato,categoria,variable,maestro,no,Contrato marco o servicio por demanda
certificacion_bpm,categoria,variable,maestro,no,Cuenta con certificación BPM (sí/no)
plan_muestreo,categoria,variable,maestro,no,Frecuencia pactada de muestreo en planta
fecha_vinculacion,fecha,fecha,maestro,no,Fecha de vinculación como cliente
distancia_km,numero,variable,maestro,no,Distancia de la planta al laboratorio (km)
canal_captacion,categoria,variable,maestro,no,Canal por el que llegó el cliente


## 3. Las tablas crudas

Se obtienen las tablas, se validan contra el contrato (si falta una columna obligatoria la etapa se
detiene con la lista completa) y se guardan tal como llegan, en CSV, de forma que la limpieza de la
etapa 02 quede como un paso aparte y auditable.

In [3]:
datos.tablas_crudas(c).mostrar()

### Cómo llegan los datos

Basta mirar las tablas para ver por qué hace falta una etapa de limpieza: la misma categoría
escrita de varias formas, recuentos reportados como texto, valores imposibles y filas repetidas.

In [4]:
datos.como_llegan(c).mostrar()

## 4. De dónde salen los datos y cómo se conecta la exportación real

La exportación real todavía no está disponible, de forma que mientras tanto las tablas salen de un
simulador que reproduce el recorrido de cada muestra por el laboratorio (con la misma «suciedad»);
las variables latentes que usa (si la muestra estaba de verdad contaminada, la higiene del cliente)
se guardan aparte en `datos/auditoria/` y ningún modelo las lee. Para usar los datos reales basta
copiar los archivos en `datos/reales/`, describir en `configuracion/mapeo_columnas.json` cómo se
llama cada columna y cada etiqueta, y fijar `PROYECTO_ORIGEN=real`, sin tocar el código.

In [5]:
datos.exportacion_real(c).mostrar()

Tabla de auditoría,Filas,Columnas latentes
verdad_muestras,38846,"contaminada_real, prob_contaminacion, prob_falso_positivo, positivo_sin_ruido, en_episodio, logit_contaminacion"
verdad_clientes,220,"segmento_latente, higiene, tasa_semanal, fecha_retiro"
verdad_ensayos,7,"ensayo, sensibilidad, especificidad_nominal, especificidad_efectiva, prevalencia_real"


## 5. Protocolo de evaluación y mapa del proyecto

La partición es por tiempo, con cortes fijados en `config` antes de mirar resultados:
**entrenamiento** para ajustar y comparar (validación cruzada de ventana creciente),
**validación** para decidir (umbral, modelo que se congela) y **prueba**, que se guarda aparte y se
abre una sola vez en la etapa 07. Las etapas solo se comunican por archivos en disco.

In [6]:
datos.protocolo(c).mostrar()

In [7]:
datos.terminar(c)

[Notebook completo](Proyecto_Final.ipynb) &nbsp;·&nbsp; Siguiente: [02 · Exploración](02_exploracion.ipynb) →